# Structured Streaming fraud detection

Author: Youssef Ibrahim Mohamed Soliman  
PySpark is optional. The cells show the event-time window and watermark used by `spark_fraud_detector.py`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, window

spark = SparkSession.builder.master("local[2]").appName("fraud-study").getOrCreate()
events = (spark.readStream
    .schema("event_id STRING, event_ts TIMESTAMP, card_id STRING, customer_id STRING, device_id STRING, country STRING, amount DOUBLE, currency STRING")
    .json("input")
    .withWatermark("event_ts", "2 minutes"))
features = events.groupBy(window(col("event_ts"), "5 minutes"), col("card_id")).agg(count("*").alias("transactions_in_window"))
features


In [ ]:
# In a real run, write to an idempotent sink and keep the checkpoint directory.
query = (features.where(col("transactions_in_window") >= 3)
    .writeStream.format("console").outputMode("append")
    .option("checkpointLocation", "checkpoints/notebook-fraud").start())
